# Clasificar textos del BOE

Ejecuta las celdas en orden. Primero predecirás la rama de un texto. Después explorarás qué grupos aparecen al agrupar textos por semejanza. [Dataset LexBOE · BSC-LT · CC BY 4.0](https://huggingface.co/datasets/BSC-LT/LexBOE)

## 1. Preparar Colab

In [ ]:
!pip install -q datasets scikit-learn pandas matplotlib seaborn joblib

In [ ]:
import json
import os
import random
import re
import time
import unicodedata
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from datasets import load_dataset

from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    adjusted_rand_score,
    classification_report,
    completeness_score,
    confusion_matrix,
    f1_score,
    homogeneity_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import normalize
from sklearn.svm import LinearSVC

print("Librerías cargadas correctamente ✅")

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

MODO_RAPIDO = True          # False = usar todo el conjunto de entrenamiento
MAX_CHARS = 3000            # recorte de cada texto
N_TRAIN_RAPIDO = 15000      # tamaño del train en modo rápido
N_NO_SUPERVISADO = 8000     # textos usados en la parte no supervisada

N_TRAIN = N_TRAIN_RAPIDO if MODO_RAPIDO else None

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

print(f"Modo rápido: {MODO_RAPIDO} | recorte: {MAX_CHARS} caracteres | semilla: {SEED}")

## 2. Leer los textos y explorar las ramas

In [ ]:
ds = load_dataset("BSC-LT/LexBOE")
print(ds)

In [ ]:
print("Columnas:", ds["train"].column_names)

ejemplo = ds["train"][0]
print("\n--- Primer registro del train ---")
print("id    :", ejemplo["id"])
print("label :", ejemplo["label"])
print("texto :", ejemplo["sentence"][:600], "...")

In [ ]:
df_train = ds["train"].to_pandas()
df_val = ds["validation"].to_pandas()
df_test = ds["test"].to_pandas()

# Por si la etiqueta viniera guardada como número (ClassLabel), la convertimos a texto.
feature_label = ds["train"].features["label"]
if hasattr(feature_label, "names"):
    for d in (df_train, df_val, df_test):
        d["label"] = d["label"].map(lambda i: feature_label.names[i])

for nombre, d in [("train", df_train), ("validation", df_val), ("test", df_test)]:
    print(f"{nombre:<11}{len(d):>8,} filas")

display(df_train.head(3))

In [ ]:
print("Valores nulos por columna (train):")
print(df_train.isna().sum().to_string())

print("\nTextos duplicados dentro del train:", int(df_train["sentence"].duplicated().sum()))

solape_ids = set(df_train["id"]) & set(df_test["id"])
solape_txt = set(df_train["sentence"]) & set(df_test["sentence"])
print("IDs repetidos entre train y test    :", len(solape_ids))
print("Textos idénticos entre train y test :", len(solape_txt))

In [ ]:
dist = pd.DataFrame({
    "train": df_train["label"].value_counts(),
    "validation": df_val["label"].value_counts(),
    "test": df_test["label"].value_counts(),
}).fillna(0).astype(int)
dist["% del train"] = (dist["train"] / dist["train"].sum() * 100).round(1)
dist = dist.sort_values("train", ascending=False)
display(dist)


# ¿Qué rama acertaría más a menudo un sistema que siempre responde lo mismo?

In [ ]:
df_train["n_palabras"] = df_train["sentence"].str.split().str.len()

display(df_train["n_palabras"].describe().round(0).to_frame("palabras por texto"))

p99 = df_train["n_palabras"].quantile(0.99)
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.hist(df_train["n_palabras"].clip(upper=p99), bins=60, color="steelblue")
ax.set_title("Distribución de longitudes (recortada al percentil 99)")
ax.set_xlabel("Palabras por texto")
ax.set_ylabel("Frecuencia")
plt.tight_layout()
plt.show()

pct_largos = (df_train["sentence"].str.len() > MAX_CHARS).mean() * 100
print(f"{pct_largos:.1f}% de los textos supera {MAX_CHARS} caracteres y será recortado en la preparación.")

## 3. Limpiar el texto y preparar las particiones

In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
ESPACIOS_RE = re.compile(r"\s+")


def limpiar_texto(texto, max_chars=MAX_CHARS):
    """Recorta, quita URLs y normaliza espacios."""
    texto = texto[:max_chars]
    texto = URL_RE.sub(" ", texto)
    texto = ESPACIOS_RE.sub(" ", texto)
    return texto.strip()

In [ ]:
for d in (df_train, df_val, df_test):
    d.dropna(subset=["sentence", "label"], inplace=True)
    d["texto"] = d["sentence"].map(limpiar_texto)

df_train = df_train[df_train["texto"].str.len() >= 20]
df_val = df_val[df_val["texto"].str.len() >= 20]
df_test = df_test[df_test["texto"].str.len() >= 20]


def submuestra_estratificada(df, n):
    """Devuelve n filas manteniendo la proporción de clases (o todo si n es None)."""
    if n is None or n >= len(df):
        return df
    # Conservamos la proporción de ramas al reducir el entrenamiento.
    muestra, _ = train_test_split(df, train_size=n, stratify=df["label"], random_state=SEED)
    return muestra


df_train_m = submuestra_estratificada(df_train, N_TRAIN)

X_train, y_train = df_train_m["texto"], df_train_m["label"]
X_val, y_val = df_val["texto"], df_val["label"]
X_test, y_test = df_test["texto"], df_test["label"]

CLASES = sorted(y_train.unique())

print(f"Train      : {len(X_train):>7,}")
print(f"Validation : {len(X_val):>7,}")
print(f"Test       : {len(X_test):>7,}")
print(f"\nClases ({len(CLASES)}): {CLASES}")

## 4. Convertir palabras en números con TF-IDF

In [ ]:
def quitar_acentos(s):
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")


STOPWORDS_ES = sorted({quitar_acentos(w) for w in """
a al algo algun alguna algunas alguno algunos ante antes aquel aquella aquellos asi aun aunque
cada como con contra cual cuales cuando de del desde donde dos durante e el ella ellas ello ellos en entre
era eran es esa esas ese eso esos esta estaba estan estar este esto estos fue fueron ha habia han hasta hay
la las le les lo los mas me mi mis mucho muy nada ni no nos nuestra nuestro o os otra otras otro otros para
pero poco por porque que quien quienes se sea sean segun ser si sido sin sobre son su sus tambien tan
te tiene tienen todo todos tu tus un una uno unos y ya
""".split()})

TFIDF_PARAMS = dict(
    lowercase=True,
    strip_accents="unicode",
    stop_words=STOPWORDS_ES,
    token_pattern=r"(?u)\b[^\W\d_]{2,}\b",
    ngram_range=(1, 1),
    min_df=5,       # La palabra debe aparecer en cinco documentos como mínimo.
    max_df=0.8,
    max_features=50000,
    sublinear_tf=True,  # Repetir una palabra aumenta su peso de forma logarítmica.
    dtype=np.float32,
)

print(f"{len(STOPWORDS_ES)} stopwords en español cargadas.")

In [ ]:
t0 = time.perf_counter()
tfidf = TfidfVectorizer(**TFIDF_PARAMS)

X_train_tfidf = tfidf.fit_transform(X_train)   # aprende vocabulario + IDF solo del train
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

print(f"Vectorización lista en {time.perf_counter() - t0:.1f} s")
print(f"Matriz de train: {X_train_tfidf.shape[0]:,} textos × {X_train_tfidf.shape[1]:,} palabras")
densidad = X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])
print(f"Densidad: {densidad:.2%} de las celdas son distintas de cero (por eso es una matriz dispersa)")

In [ ]:
terminos = tfidf.get_feature_names_out()

# Palabras con mayor peso medio en todo el train
peso_medio = np.asarray(X_train_tfidf.mean(axis=0)).ravel()
top_global = peso_medio.argsort()[::-1][:15]
print("Palabras con mayor peso TF-IDF medio:", ", ".join(terminos[top_global]))

# Palabras más importantes del primer texto
fila = X_train_tfidf[0]
orden = fila.data.argsort()[::-1][:8]
print(f"\nTexto de ejemplo (rama real: {y_train.iloc[0]}):")
print(X_train.iloc[0][:300], "...")
print("\nSus palabras más pesadas:")
for idx in fila.indices[orden]:
    print(f"  {terminos[idx]:<20} {fila[0, idx]:.3f}")

## 5. Comparar clasificadores en validación

In [ ]:
modelos = {}      # guarda cada modelo entrenado por nombre
resultados = []   # guarda las métricas de cada uno


def evaluar(nombre, modelo):
    """Entrena el modelo con train y mide su rendimiento en train y validación."""
    t0 = time.perf_counter()
    modelo.fit(X_train_tfidf, y_train)
    segundos = time.perf_counter() - t0

    pred_train = modelo.predict(X_train_tfidf)
    pred_val = modelo.predict(X_val_tfidf)

    # F1 macro da el mismo peso a cada rama, incluso si tiene pocos ejemplos.
    resultados.append({
        "modelo": nombre,
        "acc_train": accuracy_score(y_train, pred_train),
        "acc_val": accuracy_score(y_val, pred_val),
        "f1_macro_val": f1_score(y_val, pred_val, average="macro", zero_division=0),
        "f1_ponderado_val": f1_score(y_val, pred_val, average="weighted", zero_division=0),
        "segundos": segundos,
    })
    modelos[nombre] = modelo
    r = resultados[-1]
    print(f"{nombre:<34} acc_val={r['acc_val']:.3f}  F1macro_val={r['f1_macro_val']:.3f}  ({segundos:.1f} s)")

In [ ]:
evaluar("Baseline (clase más frecuente)", DummyClassifier(strategy="most_frequent"))
evaluar("Complement Naive Bayes", ComplementNB(alpha=0.3))
# balanced da más peso durante el ajuste a las clases menos frecuentes.
evaluar("Regresión logística", LogisticRegression(C=10, max_iter=300, class_weight="balanced"))
evaluar("Linear SVM (C=1)", LinearSVC(C=1.0, class_weight="balanced", random_state=SEED))

In [ ]:
tabla = pd.DataFrame(resultados).set_index("modelo").sort_values("f1_macro_val", ascending=False)
display(tabla.round(3))

In [ ]:
# Un C mayor penaliza más los errores durante el entrenamiento del SVM.
valores_C = [0.05, 0.1, 0.3, 1, 3]
f1_por_C = []
for C in valores_C:
    m = LinearSVC(C=C, class_weight="balanced", random_state=SEED)
    m.fit(X_train_tfidf, y_train)
    f1_por_C.append(f1_score(y_val, m.predict(X_val_tfidf), average="macro", zero_division=0))
    print(f"C={C:<5}  F1 macro (validación) = {f1_por_C[-1]:.4f}")

C_opt = valores_C[int(np.argmax(f1_por_C))]
print(f"\nMejor C: {C_opt}")

In [ ]:
# Elegimos entre modelos lineales para poder inspeccionar sus pesos por palabra.
evaluar(f"Linear SVM afinado (C={C_opt})", LinearSVC(C=C_opt, class_weight="balanced", random_state=SEED))

tabla = pd.DataFrame(resultados).set_index("modelo")
candidatos = tabla.loc[[n for n in tabla.index if n.startswith(("Regresión logística", "Linear SVM"))]]
mejor_nombre = candidatos["f1_macro_val"].idxmax()
mejor_modelo = modelos[mejor_nombre]

print(f"\n🏆 Mejor modelo según F1 macro en validación: {mejor_nombre}")
print(f"   F1 macro (val) = {candidatos.loc[mejor_nombre, 'f1_macro_val']:.3f}")

## 6. Entrenar el modelo final y evaluar sus predicciones

In [ ]:
X_trval = pd.concat([X_train, X_val])
y_trval = pd.concat([y_train, y_val])

pipeline_final = Pipeline([
    ("tfidf", TfidfVectorizer(**TFIDF_PARAMS)),
    ("clf", clone(mejor_modelo)),   # mismos hiperparámetros, pero sin entrenar
])

t0 = time.perf_counter()
pipeline_final.fit(X_trval, y_trval)
print(f"Pipeline final entrenado en {time.perf_counter() - t0:.1f} s")

pred_final = pipeline_final.predict(X_test)
acc_final = accuracy_score(y_test, pred_final)
f1_final = f1_score(y_test, pred_final, average="macro", zero_division=0)
print(f"\nResultado final en test → Accuracy: {acc_final:.3f} | F1 macro: {f1_final:.3f}")

In [ ]:
print(classification_report(y_test, pred_final, digits=3, zero_division=0))
# Recall indica qué fracción de los textos reales de cada rama recuperó el modelo.

In [ ]:
# Cada fila suma 1 y corresponde a una rama real.
cm = confusion_matrix(y_test, pred_final, labels=CLASES)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", cbar=False,
            xticklabels=CLASES, yticklabels=CLASES, ax=ax, annot_kws={"size": 8})
ax.set_xlabel("Rama predicha")
ax.set_ylabel("Rama real")
ax.set_title(f"Matriz de confusión normalizada: {mejor_nombre}")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
mask_err = pred_final != y_test.values
idx_err = np.where(mask_err)[0][:5]
print(f"Total de errores en test: {mask_err.sum():,} de {len(y_test):,}\n")
for i in idx_err:
    print(f"Real: {y_test.iloc[i]}  |  Predicho: {pred_final[i]}")
    print("   ", X_test.iloc[i][:300], "...\n")

In [ ]:
# Un peso positivo alto favorece esa rama cuando el término aparece en el texto.
terminos = np.array(pipeline_final.named_steps["tfidf"].get_feature_names_out())
coef = pipeline_final.named_steps["clf"].coef_

top_terminos = {}
for i, clase in enumerate(pipeline_final.named_steps["clf"].classes_):
    top_idx = coef[i].argsort()[::-1][:10]
    top_terminos[clase] = list(terminos[top_idx])

tabla_terminos = pd.DataFrame(top_terminos).T
tabla_terminos.columns = [f"#{i + 1}" for i in range(tabla_terminos.shape[1])]
display(tabla_terminos)

## 7. Guardar y probar con tu propio texto

In [ ]:
RUTA_MODELO = "modelo_lexboe.joblib"
RUTA_INFO = "modelo_lexboe_info.json"

joblib.dump(pipeline_final, RUTA_MODELO, compress=3)
print(f"Modelo guardado: {RUTA_MODELO} ({os.path.getsize(RUTA_MODELO) / 1e6:.1f} MB)")

metadatos = {
    "dataset": "BSC-LT/LexBOE",
    "fecha": time.strftime("%Y-%m-%d %H:%M"),
    "modelo": mejor_nombre,
    "hiperparametros": pipeline_final.named_steps["clf"].get_params(),
    "clases": list(pipeline_final.classes_),
    "accuracy_test": round(float(acc_final), 4),
    "f1_macro_test": round(float(f1_final), 4),
    "modo_rapido": MODO_RAPIDO,
    "textos_entrenamiento": int(len(X_trval)),
    "max_chars": MAX_CHARS,
}
with open(RUTA_INFO, "w", encoding="utf-8") as f:
    json.dump(metadatos, f, ensure_ascii=False, indent=2, default=str)
print(f"Información guardada: {RUTA_INFO}")

In [ ]:
modelo_cargado = joblib.load(RUTA_MODELO)

muestra_check = X_test.iloc[:200]
assert (modelo_cargado.predict(muestra_check) == pipeline_final.predict(muestra_check)).all()
print("✅ El modelo cargado desde disco predice igual que el original.")
print("Clases que conoce:", list(modelo_cargado.classes_))

In [ ]:
# Cambia esta frase y ejecuta otra vez. El modelo se entrenó con textos limpios.
texto = "Se aprueba el plan de estudios de un nuevo grado universitario."
entrada = [limpiar_texto(texto)]
print("Rama predicha:", modelo_cargado.predict(entrada)[0])

## 8. Descubrir grupos con K-Means

In [ ]:
muestra_ns = df_train.sample(n=min(N_NO_SUPERVISADO, len(df_train)), random_state=SEED)
textos_ns = muestra_ns["texto"]
etiquetas_ref = muestra_ns["label"].values       # NO se usan para entrenar

vec_ns = TfidfVectorizer(**{**TFIDF_PARAMS, "max_features": 20000})
X_ns = vec_ns.fit_transform(textos_ns)

# SVD resume las columnas TF-IDF en 100 componentes antes de calcular distancias.
svd = TruncatedSVD(n_components=100, random_state=SEED)
X_# SVD resume las columnas TF-IDF en 100 componentes antes de calcular distancias.
svd = normalize(svd.fit_transform(X_ns))

print(f"Textos usados: {X_ns.shape[0]:,} | palabras: {X_ns.shape[1]:,}")
print(f"Tras SVD: {X_svd.shape[1]} dimensiones, que conservan {svd.explained_variance_ratio_.sum():.1%} de la varianza")

In [ ]:
rango_k = list(range(2, 17))
inercias, siluetas = [], []
for k in rango_k:
    km = KMeans(n_clusters=k, n_init=3, random_state=SEED)
    etiquetas_k = km.fit_predict(X_svd)
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(X_svd, etiquetas_k,
                                     sample_size=min(3000, X_svd.shape[0]), random_state=SEED))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(rango_k, inercias, marker="o", color="steelblue")
axes[0].set_title("Método del codo")
axes[0].set_xlabel("K (número de grupos)")
axes[0].set_ylabel("Inercia")
axes[1].plot(rango_k, siluetas, marker="o", color="darkorange")
axes[1].set_title("Silhouette score")
axes[1].set_xlabel("K (número de grupos)")
axes[1].set_ylabel("Silhouette")
plt.tight_layout()
plt.show()

In [ ]:
# Usamos tantos grupos como ramas para facilitar la comparación posterior.
K = len(CLASES)
kmeans = KMeans(n_clusters=K, n_init=10, random_state=SEED)
clusters = kmeans.fit_predict(X_svd)

print("Tamaño de cada grupo:")
print(pd.Series(clusters).value_counts().sort_index().to_string())

# Palabras más características de cada grupo (mayor TF-IDF medio dentro del grupo)
terminos_ns = np.array(vec_ns.get_feature_names_out())
filas_terminos = {}
for c in range(K):
    media_c = np.asarray(X_ns[clusters == c].mean(axis=0)).ravel()
    filas_terminos[f"Grupo {c}"] = list(terminos_ns[media_c.argsort()[::-1][:8]])

print("\nPalabras más características de cada grupo:")
display(pd.DataFrame(filas_terminos).T)

In [ ]:
# Ahora sí usamos las etiquetas reales para medir cuánto coinciden con los grupos.
ari = adjusted_rand_score(etiquetas_ref, clusters)
nmi = normalized_mutual_info_score(etiquetas_ref, clusters)
hom = homogeneity_score(etiquetas_ref, clusters)
comp = completeness_score(etiquetas_ref, clusters)

tabla_cruzada = pd.crosstab(pd.Series(clusters, name="grupo"), pd.Series(etiquetas_ref, name="rama real"))
pureza = tabla_cruzada.max(axis=1).sum() / tabla_cruzada.values.sum()

print(f"ARI           : {ari:.3f}")
print(f"NMI           : {nmi:.3f}")
print(f"Homogeneidad  : {hom:.3f}")
print(f"Completitud   : {comp:.3f}")
print(f"Pureza        : {pureza:.3f}")

cruzada_norm = tabla_cruzada.div(tabla_cruzada.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(11, 6.5))
sns.heatmap(cruzada_norm, annot=True, fmt=".2f", cmap="YlGnBu", cbar=False, ax=ax, annot_kws={"size": 8})
ax.set_title("¿Qué ramas reales hay dentro de cada grupo? (cada fila suma 1)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
coords = PCA(n_components=2, random_state=SEED).fit_transform(X_svd)
idx_plot = np.random.RandomState(SEED).choice(len(coords), size=min(3000, len(coords)), replace=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.scatterplot(x=coords[idx_plot, 0], y=coords[idx_plot, 1], hue=clusters[idx_plot].astype(str),
                palette="tab20", s=12, linewidth=0, ax=axes[0], legend=False)
axes[0].set_title("Coloreado por grupo de K-Means (sin etiquetas)")

sns.scatterplot(x=coords[idx_plot, 0], y=coords[idx_plot, 1], hue=etiquetas_ref[idx_plot],
                palette="tab20", s=12, linewidth=0, ax=axes[1])
axes[1].set_title("Coloreado por rama real")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7, markerscale=2)

for ax in axes:
    ax.set_xlabel("Componente 1")
    ax.set_ylabel("Componente 2")
plt.tight_layout()
plt.show()

**Para interpretar el resultado.** Elige un grupo, lee sus términos y localízalo en la tabla de grupos y ramas. ¿Qué nombre le darías?